## **Uso de Snowpark**

In [ ]:
import streamlit as st
import pandas as pd
import os
import configparser
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, sum

# Configuración de la conexión a Snowflake.
config_path = os.path.join(os.environ['USERPROFILE'], '.snowsql', 'config')
config = configparser.ConfigParser()
config.read(config_path)

# Se obtienen los parámetros de conexión desde el archivo de configuración.
try:
    account = config['connections.example']['accountname']
    user = config['connections.example']['username']
    password = config['connections.example']['password']
except KeyError as e:
    print(f'Error: {e}')

# Se definen los parámetros de conexión a Snowflake.
connection_parameters = {
    "account": account,
    "user": user,
    "password": password,
    "warehouse": "COMPUTE_WH",
    "database": "DEVMOON_SAMPLE",
    "schema": "PUBLIC",
}

# Se crea la sesión de Snowpark utilizando los parámetros de conexión.
try:
    session = Session.builder.configs(connection_parameters).create()
    print('Conexión exitosa con Snowpark.')
except Exception as e:
    print(f'Error: {e}')

Conexión exitosa con Snowpark.


In [ ]:
# Se selecciona la tabla "CAMPAING_DATA_PYTHON" y se obtienen sus columnas.

df = session.table("CAMPAING_DATA_PYTHON")
df.columns

['DATESTART', 'CAMPAING', 'REGION', 'CLICKS', 'IMPRESSIONS', 'VIEWS', 'COST']

In [ ]:
# Se selecciona la columna "Region" de la tabla y se muestran los resultados.

seleccionar = df.select('Region')
seleccionar.show()

------------
|"REGION"  |
------------
|Oeste     |
|Norte     |
|Norte     |
|Sur       |
|Este      |
|Norte     |
|Norte     |
|Oeste     |
|Oeste     |
|Este      |
------------



In [ ]:
# Se agrupan los datos por la columna "Region" y se calcula la suma de la columna "Views" para cada región.

vistas_por_region = df.group_by("Region").agg(
    sum(col("Views")).alias(
        "TOTAL_VISTAS"
        )
    )
vistas_por_region.show()

-----------------------------
|"REGION"  |"TOTAL_VISTAS"  |
-----------------------------
|Norte     |6731836         |
|Sur       |6804491         |
|Oeste     |7029798         |
|Este      |6549980         |
-----------------------------



In [ ]:
# Se agrupan los datos por la columna "Region" y se calcula la suma de la columna "Clicks" para cada región.

clicks_por_region = df.group_by("Region").agg(sum(col("Clicks")).alias("TOTAL_CLICKS"))
clicks_por_region.show()

-----------------------------
|"REGION"  |"TOTAL_CLICKS"  |
-----------------------------
|Norte     |6724107         |
|Este      |6948197         |
|Oeste     |7297555         |
|Sur       |7100654         |
-----------------------------



In [ ]:
# Se filtran los datos para mostrar solo las filas donde la columna "Impressions" sea mayor a 100,000.

filtro = df.filter('Impressions > 100000')
filtro.show()

-----------------------------------------------------------------------------------------------
|"DATESTART"  |"CAMPAING"            |"REGION"  |"CLICKS"  |"IMPRESSIONS"  |"VIEWS"  |"COST"  |
-----------------------------------------------------------------------------------------------
|2023-12-24   |CursosDeProgramacion  |Oeste     |26148     |133468         |21979    |606     |
|2023-11-11   |AprendeCSharpFacil    |Norte     |34586     |340290         |49146    |495     |
|2024-05-24   |CodigoEnEspanol       |Norte     |20337     |343710         |32831    |1655    |
|2023-09-05   |AprendePythonFacil    |Sur       |45332     |279000         |15840    |991     |
|2024-06-19   |CodigoEnEspanol       |Este      |36817     |462135         |42041    |1784    |
|2023-08-22   |AprendeJavaScriptYA   |Norte     |39359     |440556         |5551     |205     |
|2024-04-10   |AprendeJavaScriptYA   |Norte     |24900     |338969         |8259     |134     |
|2023-10-06   |AprendeCSharpFacil    |Oe